In [ ]:
#@title ① git setup — run once per session
# Clones the repo to Drive on first run, then just pulls on subsequent sessions.
# Drive persists between Colab sessions so you never re-clone.

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = "/content/drive/MyDrive/SODA-policy"  # change if you want a different Drive path
REPO_URL = "https://github.com/umar-padela/SODA-policy.git"
BRANCH   = "dev_umar"

import os
if not os.path.exists(REPO_DIR):
    print("Cloning repo to Drive (one-time, ~30s)...")
    !git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"
else:
    print("Repo already on Drive. Pulling latest...")
    !git -C "{REPO_DIR}" pull origin {BRANCH}

# Configure git identity so commits work from Colab
!git -C "{REPO_DIR}" config user.email "umarpadela@gmail.com"
!git -C "{REPO_DIR}" config user.name  "umar-padela"

print(f"\nRepo ready at {REPO_DIR}")

In [ ]:
#@title ② pull latest changes — re-run to sync with the latest version on git
# Run this cell, then File → Open notebook → navigate to it in Drive to reload.
# Your runtime variables (episode_files, client, etc.) stay alive — you only lose
# the notebook code itself, which is what you just pulled.

!git -C "{REPO_DIR}" pull origin {BRANCH}
print("\nDone. If the notebook changed: File → Open notebook → navigate to it in Drive.")
print(f"Path: {REPO_DIR}/soda/option_discovery/supervised/D5RL_kitchen/explore_and_label.ipynb")

In [ ]:
#@title ③ installs
!pip install -q google-genai mediapy opencv-python-headless numpy matplotlib

In [ ]:
#@title ④ config — EDIT THIS CELL

# --- Episode to explore ---
EPISODE_IDX = 0  #@param {type: "integer"}

# --- Camera to use for labeling (D5RL kitchen has two: 'image' and 'wrist_image') ---
CAMERA = "image"  #@param ["image", "wrist_image"]

# --- FPS the video was recorded at (D5RL kitchen is ~10 Hz) ---
DATA_FPS = 10  #@param {type: "integer"}

# --- Frame sampling rate sent to Gemini ---
# Lower = more detail, higher = faster/cheaper.
GEMINI_VIDEO_FPS = 5  #@param {type: "integer"}

# --- Gemini model ---
GEMINI_MODEL = "gemini-2.5-pro-preview-05-06"  #@param ["gemini-2.5-pro-preview-05-06", "gemini-2.5-flash-preview-04-17", "gemini-2.0-flash"]

# --- Where D5RL data lives on Drive ---
DRIVE_DATA_DIR = "/content/drive/MyDrive/d5rl_kitchen"

# --- Prompt (edit freely — this is the thing you'll iterate on) ---
KITCHEN_PROMPT = """
Role: Robotics Data Specialist
Task: Segment a robot kitchen manipulation demonstration into a sequence of discrete options.

## Video Context ##
A Franka robot arm is performing a series of kitchen manipulation tasks.
The robot must interact with up to 4 objects in some order: the microwave, the kettle,
the light switch, and the sliding cabinet door.

**Environment Elements:**
- **Franka Robot Arm**: A robot arm visible in the scene.
- **Microwave**: A microwave with a door that swings open.
- **Kettle**: A kettle sitting on the stovetop.
- **Light Switch**: A toggle switch on the back wall.
- **Sliding Cabinet**: A cabinet door that slides open horizontally.

**Option Definitions:**
1. **TRANSIT**: The robot arm is moving through free space between objects. No object is being manipulated.
2. **MICROWAVE**: The robot is actively reaching toward or interacting with the microwave door.
3. **KETTLE**: The robot is actively reaching toward, grasping, or moving the kettle.
4. **LIGHT_SWITCH**: The robot is actively reaching toward or flipping the light switch.
5. **CABINET**: The robot is actively reaching toward or sliding open the cabinet door.

**Frame Number Tracking:**
Each frame has a red number burned into the top-left corner. Read these directly — do not estimate.

**Processing Workflow:**
1. Watch the full video and write a chain-of-thought describing the sequence of options.
2. Identify the exact frame number where each transition occurs.
3. Self-check: does every frame belong to exactly one option? No gaps or overlaps.
4. Output the JSON array.

**Output Format:**
Two sections separated by `---`. First: chain of thought + frame ranges. Second: JSON array.

### Example Output:

Chain of thought:
The robot starts in free space (TRANSIT). It then moves to and opens the microwave (MICROWAVE).
It pulls back briefly (TRANSIT), then picks up the kettle (KETTLE).
It returns to free space (TRANSIT) and flips the light switch (LIGHT_SWITCH).
Finally it slides the cabinet open (CABINET).

TRANSIT: 0-24
MICROWAVE: 24-67
TRANSIT: 67-89
KETTLE: 89-134
TRANSIT: 134-151
LIGHT_SWITCH: 151-178
CABINET: 178-220

---
[
  {"option": "TRANSIT",      "start": 0,   "end": 24},
  {"option": "MICROWAVE",    "start": 24,  "end": 67},
  {"option": "TRANSIT",      "start": 67,  "end": 89},
  {"option": "KETTLE",       "start": 89,  "end": 134},
  {"option": "TRANSIT",      "start": 134, "end": 151},
  {"option": "LIGHT_SWITCH", "start": 151, "end": 178},
  {"option": "CABINET",      "start": 178, "end": 220}
]
"""

In [ ]:
#@title ⑤ imports + API key setup
import os
import glob
import time
import json
import math
import numpy as np
import cv2
import matplotlib.pyplot as plt
import mediapy as media
from IPython.display import display, Markdown
from google import genai
from google.genai import types
from google.colab import userdata

# Add Gemini API key via the 🔑 icon in the left sidebar → name it GEMINI_API_KEY
os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
client = genai.Client()
print("Gemini client ready.")

In [ ]:
#@title ⑥ download D5RL kitchen data (Drive already mounted by cell ①)
# Only downloads once — subsequent runs just confirm data is present.

os.makedirs(DRIVE_DATA_DIR, exist_ok=True)

npz_dir = os.path.join(DRIVE_DATA_DIR, "kitchen_demos_multitask_lexa_view_and_wrist_npz")

if not os.path.exists(npz_dir) or len(glob.glob(os.path.join(npz_dir, "*.npz"))) == 0:
    print("Downloading D5RL kitchen demos from GCS (this takes a few minutes)...")
    !gsutil -m cp -r "gs://d5rl_datasets/KITCHEN_DATA/kitchen_demos_multitask_lexa_view_and_wrist_npz" "{DRIVE_DATA_DIR}/"
    print("Download complete.")
else:
    print(f"Data already present at {npz_dir}")

episode_files = sorted(glob.glob(os.path.join(npz_dir, "*.npz")))
print(f"Found {len(episode_files)} episode files.")

In [ ]:
#@title ⑦ explore data format (run once to understand the schema)
sample = np.load(episode_files[0], allow_pickle=True)
print("Top-level keys:", list(sample.keys()))
print()
for k in sample.keys():
    v = sample[k]
    if hasattr(v, 'shape'):
        print(f"  {k}: shape={v.shape}, dtype={v.dtype}")
    elif isinstance(v, dict) or (hasattr(v, 'item') and isinstance(v.item(), dict)):
        inner = v.item() if hasattr(v, 'item') else v
        print(f"  {k}: dict with keys={list(inner.keys())}")
        for ik, iv in inner.items():
            if hasattr(iv, 'shape'):
                print(f"    {ik}: shape={iv.shape}, dtype={iv.dtype}")
    else:
        print(f"  {k}: {type(v)}")

In [ ]:
#@title ⑧ helper functions

OPTION_NAMES  = ["TRANSIT", "MICROWAVE", "KETTLE", "LIGHT_SWITCH", "CABINET"]
OPTION_COLORS = {
    "TRANSIT":      (128, 128, 128),
    "MICROWAVE":    (255,  60,  60),
    "KETTLE":       ( 60, 200,  60),
    "LIGHT_SWITCH": (255, 165,   0),
    "CABINET":      (120,  80, 200),
}


def load_episode(episode_files, idx):
    data = np.load(episode_files[idx], allow_pickle=True)
    if 'observations' in data:
        obs_raw = data['observations']
        if obs_raw.dtype == object:
            obs    = obs_raw.item()
            images = obs.get('image', obs.get('pixels', obs.get('rgb', None)))
            wrist  = obs.get('wrist_image', obs.get('hand_image', None))
            state  = obs.get('state', obs.get('proprio', None))
        else:
            images, wrist, state = obs_raw, None, None
    elif 'image' in data:
        images = data['image']
        wrist  = data.get('wrist_image', None)
        state  = data.get('state', None)
    else:
        raise ValueError(f"Unrecognized npz format. Keys: {list(data.keys())}")
    actions = data.get('actions', data.get('action', None))
    return images, wrist, state, actions


def burn_frame_number(frame, idx):
    out = frame.copy()
    h, w = out.shape[:2]
    scale = max(0.4, w / 200)
    cv2.putText(out, str(idx), (4, int(14 * scale)),
                cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 0, 0), 1, cv2.LINE_AA)
    return out


def burn_option_overlay(frame, option_name):
    out = frame.copy()
    color = OPTION_COLORS.get(option_name, (255, 255, 255))
    h, w = out.shape[:2]
    cv2.rectangle(out, (0, 0), (w - 1, h - 1), color, 3)
    scale = max(0.35, w / 250)
    cv2.putText(out, option_name, (4, h - 6),
                cv2.FONT_HERSHEY_SIMPLEX, scale, color, 1, cv2.LINE_AA)
    return out


def frames_to_mp4(frames, fps, path, burn_numbers=True, option_labels=None):
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    for i, f in enumerate(frames):
        out = f.copy()
        if burn_numbers:
            out = burn_frame_number(out, i)
        if option_labels is not None and i < len(option_labels):
            out = burn_option_overlay(out, option_labels[i])
        writer.write(cv2.cvtColor(out, cv2.COLOR_RGB2BGR))
    writer.release()


def play_episode(episode_files, idx, camera='image', height=400):
    images, wrist, state, actions = load_episode(episode_files, idx)
    frames = images if camera == 'image' else (wrist if wrist is not None else images)
    if frames is None:
        raise ValueError(f"Camera '{camera}' not found.")
    if frames.dtype != np.uint8:
        frames = (frames * 255).clip(0, 255).astype(np.uint8)
    print(f"Episode {idx} | {len(frames)} frames | action_dim={actions.shape[-1] if actions is not None else 'N/A'}")
    if state is not None:
        print(f"State dim: {state.shape[-1]}")
    tmp = f"_ep{idx}_raw.mp4"
    frames_to_mp4(list(frames), fps=DATA_FPS, path=tmp, burn_numbers=True)
    media.show_video(media.read_video(tmp), height=height)
    os.remove(tmp)


def segments_to_frame_labels(segments, n_frames):
    labels = ["TRANSIT"] * n_frames
    for seg in segments:
        for i in range(seg['start'], min(seg['end'], n_frames)):
            labels[i] = seg['option']
    return labels


def parse_json_from_response(response_text):
    parts = response_text.split('---')
    json_part = parts[-1].strip().replace('```json', '').replace('```', '').strip()
    return json.loads(json_part)


print("Helper functions loaded.")

In [ ]:
#@title ⑨ play raw episode (no labels)
play_episode(episode_files, EPISODE_IDX, camera=CAMERA, height=400)

In [ ]:
#@title ⑩ label episode with Gemini
# Edit KITCHEN_PROMPT in cell ④, then re-run this cell.

def label_episode_gemini(episode_files, idx, camera, prompt, model, video_fps):
    images, wrist, state, actions = load_episode(episode_files, idx)
    frames = images if camera == 'image' else (wrist if wrist is not None else images)
    if frames.dtype != np.uint8:
        frames = (frames * 255).clip(0, 255).astype(np.uint8)
    n_frames = len(frames)
    tmp_path = f"_label_ep{idx}.mp4"

    print(f"Compiling {n_frames}-frame video for episode {idx}...")
    frames_to_mp4(list(frames), fps=DATA_FPS, path=tmp_path, burn_numbers=True)

    try:
        print("Uploading to Gemini File API...")
        cloud_file = client.files.upload(file=tmp_path)
        while cloud_file.state.name == "PROCESSING":
            print(".", end="", flush=True)
            time.sleep(4)
            cloud_file = client.files.get(name=cloud_file.name)
        if cloud_file.state.name == "FAILED":
            raise RuntimeError("Gemini file processing failed.")

        print(f"\nFile ready. Calling {model}...")
        response = client.models.generate_content(
            model=model,
            contents=types.Content(
                parts=[
                    types.Part(
                        file_data=types.FileData(file_uri=cloud_file.uri, mime_type="video/mp4"),
                        video_metadata=types.VideoMetadata(fps=video_fps)
                    ),
                    types.Part(text=prompt)
                ]
            ),
            config=types.GenerateContentConfig(
                temperature=0.0,
                thinking_config=types.ThinkingConfig(include_thoughts=True, thinking_budget=4000),
            )
        )

        print("\n--- Gemini Response ---")
        display(Markdown(response.text))
        print(f"\nTokens — prompt: {response.usage_metadata.prompt_token_count}, "
              f"thoughts: {response.usage_metadata.thoughts_token_count}, "
              f"output: {response.usage_metadata.candidates_token_count}")
        return response.text, n_frames

    finally:
        if 'cloud_file' in locals():
            client.files.delete(name=cloud_file.name)
            print("Remote file deleted.")
        if os.path.exists(tmp_path):
            os.remove(tmp_path)


response_text, n_frames = label_episode_gemini(
    episode_files, EPISODE_IDX, CAMERA, KITCHEN_PROMPT, GEMINI_MODEL, GEMINI_VIDEO_FPS
)

In [ ]:
#@title ⑪ visualize labeled episode

try:
    segments = parse_json_from_response(response_text)
    print("Parsed segments:")
    for s in segments:
        print(f"  {s['option']:15s}  frames {s['start']:4d} – {s['end']:4d}  ({s['end']-s['start']} frames)")
except Exception as e:
    print(f"Could not parse JSON automatically: {e}")
    print("Paste the JSON array manually into 'segments' below and re-run.")
    segments = []  # <- paste manually if needed

if segments:
    images, wrist, state, actions = load_episode(episode_files, EPISODE_IDX)
    frames = images if CAMERA == 'image' else (wrist if wrist is not None else images)
    if frames.dtype != np.uint8:
        frames = (frames * 255).clip(0, 255).astype(np.uint8)
    per_frame_labels = segments_to_frame_labels(segments, len(frames))
    tmp = f"_labeled_ep{EPISODE_IDX}.mp4"
    frames_to_mp4(list(frames), fps=DATA_FPS, path=tmp, burn_numbers=True, option_labels=per_frame_labels)
    print(f"\nLabeled episode {EPISODE_IDX}:")
    media.show_video(media.read_video(tmp), height=400)
    os.remove(tmp)

In [ ]:
#@title ⑫ option distribution for this episode

if segments:
    from collections import Counter
    counts = Counter(per_frame_labels)
    total  = sum(counts.values())
    print(f"{'Option':<15} {'Frames':>8} {'%':>8}")
    print("-" * 35)
    for name in OPTION_NAMES:
        n = counts.get(name, 0)
        print(f"{name:<15} {n:>8}  {100*n/total:>6.1f}%")
    fig, ax = plt.subplots(figsize=(8, 3))
    colors = [tuple(c/255 for c in OPTION_COLORS[n]) for n in OPTION_NAMES]
    ax.bar(OPTION_NAMES, [counts.get(n, 0) for n in OPTION_NAMES], color=colors, edgecolor='black')
    ax.set_ylabel("Frames")
    ax.set_title(f"Option distribution — Episode {EPISODE_IDX}")
    plt.tight_layout()
    plt.show()

In [ ]:
#@title ⑬ push your changes back to git — run when you want to save notebook edits

NOTEBOOK_REL = "soda/option_discovery/supervised/D5RL_kitchen/explore_and_label.ipynb"
COMMIT_MSG   = "notebook: update from Colab session"  #@param {type: "string"}

!git -C "{REPO_DIR}" add "{NOTEBOOK_REL}"
!git -C "{REPO_DIR}" diff --cached --stat
!git -C "{REPO_DIR}" commit -m "{COMMIT_MSG}" || echo "Nothing to commit."
!git -C "{REPO_DIR}" push origin {BRANCH}